# Init

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.table('projeto_combustiveis.bronze.semanal_capitais')
df.display()

# DICIONÁRIO DADOS SEMANAIS

In [0]:
dicionario_colunas_semanais = {
    "regio": "regiao",
    "estados": "estado", # Corrige a inconsistência da ANP (plural para singular)
    "municpio": "municipio",
    "nmero_de_postos_pesquisados": "numero_postos_pesquisados",
    "preo_mdio_revenda": "preco_medio_revenda",
    "desvio_padro_revenda": "desvio_padrao_revenda",
    "preo_mnimo_revenda": "preco_minimo_revenda",
    "preo_mximo_revenda": "preco_maximo_revenda",
    "coef_de_variao_revenda": "coef_variacao_revenda"
}

# FUNÇÃO DE LIMPEZA E PADRONIZAÇÃO DA SILVER LAYER




In [0]:
def limpa_tabela_semanal(nome_tabela_bronze, nome_tabela_silver, dict_renomear):
    print(f"A iniciar o processamento da tabela: {nome_tabela_silver}...")
    
    # Leitura da tabela Bruta
    df = spark.read.table(f"projeto_combustiveis.bronze.{nome_tabela_bronze}")
    
    # Passo 1: Renomear colunas corrompidas e padronizar o singular/plural da ANP
    for antiga, nova in dict_renomear.items():
        if antiga in df.columns:
            df = df.withColumnRenamed(antiga, nova)
            
    # Passo 2: Aplicar TRIM
    colunas_texto = [coluna for coluna, tipo in df.dtypes if tipo == "string"]
    for col in colunas_texto:
        df = df.withColumn(col, F.trim(F.col(col)))
        
    # Passo 3: O Truque da Data de Referência e tipagem de datas
    if "data_inicial" in df.columns:
        df = df.withColumn("data_inicial", F.to_date(F.col("data_inicial"), "yyyy-MM-dd"))
    if "data_final" in df.columns:
        df = df.withColumn("data_final", F.to_date(F.col("data_final"), "yyyy-MM-dd"))
        # Criando a coluna universal para o Power BI
        df = df.withColumn("data_referencia", F.col("data_final"))
        
    # Passo 4: Tratar nulos ("-") e tipar a quantidade de postos (Inteiro)
    if "numero_postos_pesquisados" in df.columns:
        df = df.withColumn(
            "numero_postos_pesquisados", 
            F.when(F.col("numero_postos_pesquisados") == "-", None).otherwise(F.col("numero_postos_pesquisados")).cast("int")
        )
        
    # Passo 5: Tratar nulos ("-") e tipar dinamicamente as colunas financeiras (Decimal/Double)
    palavras_chave_numericas = ["preco", "desvio", "coef"]
    colunas_decimais = [c for c in df.columns if any(palavra in c for palavra in palavras_chave_numericas)]
    
    for col in colunas_decimais:
        df = df.withColumn(
            col,
            F.when(F.col(col) == "-", None).otherwise(F.col(col)).cast("double")
        )
        
    # Passo 6: Qualidade de Dados
    df = df.dropDuplicates()
    
    if "data_referencia" in df.columns and "produto" in df.columns:
        df = df.dropna(subset=["data_referencia", "produto"])
        
    # Passo 7: Governança
    df = df.withColumn("data_processamento_silver", F.current_timestamp())
    
    # Salvar na Camada silver
    df.write.mode("overwrite").saveAsTable(f"projeto_combustiveis.silver.{nome_tabela_silver}")
    print(f"Tabela {nome_tabela_silver} guardada com sucesso no Unity Catalog!\n")

# EXECUÇÃO DO PIPELINE PARA TABELAS DE MONITORAMENTO

In [0]:
limpa_tabela_semanal("semanal_brasil", "monitor_semanal_brasil", dicionario_colunas_semanais)
limpa_tabela_semanal("semanal_regioes", "monitor_semanal_regioes", dicionario_colunas_semanais)
limpa_tabela_semanal("semanal_estados", "monitor_semanal_estados", dicionario_colunas_semanais)
limpa_tabela_semanal("semanal_municipios", "monitor_semanal_municipios", dicionario_colunas_semanais)
limpa_tabela_semanal("semanal_capitais", "monitor_semanal_capitais", dicionario_colunas_semanais)

# VERIFICAÇÃO EM SQL


In [0]:
%sql
select * from projeto_combustiveis.silver.monitor_semanal_brasil limit(10)

In [0]:
%sql
select * from projeto_combustiveis.silver.monitor_semanal_regioes limit(10)

In [0]:
%sql
select * from projeto_combustiveis.silver.monitor_semanal_estados limit(10)

In [0]:
%sql
select * from projeto_combustiveis.silver.monitor_semanal_capitais limit(10)

In [0]:
%sql
select * from projeto_combustiveis.silver.monitor_semanal_municipios limit(10)